<a href="https://colab.research.google.com/github/sophieelin/multimodal_ai/blob/main/mmai_midterm_report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Data Collection

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import json
import time
import os
import requests
import urllib.parse
import pandas as pd
from pathlib import Path

# api keys
HASDATA_KEY = userdata.get('HASDATA_KEY')
GOOGLE_API_KEY = userdata.get('api_key')
CENSUS_KEY = userdata.get('CENSUS_KEY')

# paths
MASTER_CSV = '/content/drive/MyDrive/mmai_midterm_report/boston_listings_with_census.csv'
GSV_DIR = Path('/content/drive/MyDrive/mmai_midterm_report/street_view')
SAT_DIR = Path('/content/drive/MyDrive/mmai_midterm_report/satellite')
GSV_DIR.mkdir(exist_ok=True)
SAT_DIR.mkdir(exist_ok=True)


Load exisiting data

In [ ]:
if os.path.exists(MASTER_CSV):
    existing_df = pd.read_csv(MASTER_CSV, dtype={'id': str})
    seen_ids = set(existing_df['id'].astype(str))
    print(f"Loaded {len(seen_ids)} existing properties")
else:
    existing_df = pd.DataFrame()
    seen_ids = set()
    print("No existing data found")

Fetching listings from hasdata API

In [ ]:
def fetch_zillow_listings(keyword, listing_type, page=1):
    encoded_keyword = urllib.parse.quote(keyword)
    url = (
        f"https://api.hasdata.com/scrape/zillow/listing"
        f"?keyword={encoded_keyword}&type={listing_type}&page={page}"
    )
    headers = {
        'Content-Type': 'application/json',
        'x-api-key': HASDATA_KEY
    }
    r = requests.get(url, headers=headers, timeout=30)
    print(f"  Page {page} status: {r.status_code}")
    if r.status_code == 200:
        return r.json()
    return {}


def fetch_all_new_listings(keyword='Boston, MA', listing_type='sold', max_pages=20):
    all_new_props = []
    for page in range(1, max_pages + 1):
        result = fetch_zillow_listings(keyword, listing_type, page=page)
        props = result.get('properties', [])
        if not props:
            print(f"No results on page {page}")
            break
        new_props = [p for p in props if str(p['id']) not in seen_ids]
        print(f"Page {page}: {len(props)} total, {len(new_props)} new")
        all_new_props.extend(new_props)
        time.sleep(1.0)
    return all_new_props

Parsing the properties

In [ ]:
def parse_properties(props):
    rows = []
    for p in props:
        rows.append({
            'id': str(p['id']),
            'url': p['url'],
            'home_type': p['homeType'],
            'status': p['status'],
            'price': p.get('price'),
            'zestimate': p.get('zestimate'),
            'days_on_market': p.get('daysOnZillow'),
            'area_sqft': p.get('area'),
            'beds': p.get('beds'),
            'baths': p.get('baths'),
            'lat': p.get('latitude'),
            'lon': p.get('longitude'),
            'street': p['address']['street'],
            'city': p['address']['city'],
            'zipcode': p['address']['zipcode'],
            'broker': p.get('brokerName'),
            'photo_urls': p.get('photos', []),
            'first_photo': p['photos'][0] if p.get('photos') else None,
        })
    return rows

Google Street View

In [ ]:
def fetch_street_view(lat, lon, prop_id, heading=0, size='640x640', fov=90):
    path = GSV_DIR / f"{prop_id}_h{heading}.jpg"
    if path.exists():
        return str(path)
    url = (
        f"https://maps.googleapis.com/maps/api/streetview"
        f"?size={size}&location={lat},{lon}"
        f"&heading={heading}&fov={fov}&pitch=0&source=outdoor&key={GOOGLE_API_KEY}"
    )
    r = requests.get(url, timeout=10)
    if r.status_code == 200 and r.headers.get('content-type', '').startswith('image'):
        with open(path, 'wb') as f:
            f.write(r.content)
        return str(path)
    return None


def fetch_all_street_views(df):
    gsv_paths = {}
    for _, row in df.iterrows():
        paths = {}
        for heading in [0, 90, 180, 270]:
            p = fetch_street_view(row['lat'], row['lon'], row['id'], heading=heading)
            paths[heading] = p
            time.sleep(0.1)
        gsv_paths[row['id']] = paths
        print(f"  GSV done: {row['street']}")
    return gsv_paths

Satellite imagery

In [ ]:
def fetch_satellite(lat, lon, prop_id, zoom=18, size='640x640'):
    path = SAT_DIR / f"{prop_id}_sat.jpg"
    if path.exists():
        return str(path)
    url = (
        f"https://maps.googleapis.com/maps/api/staticmap"
        f"?center={lat},{lon}&zoom={zoom}"
        f"&size={size}&maptype=satellite&key={GOOGLE_API_KEY}"
    )
    r = requests.get(url, timeout=10)
    if r.status_code == 200 and r.headers.get('content-type', '').startswith('image'):
        with open(path, 'wb') as f:
            f.write(r.content)
        return str(path)
    return None

def fetch_all_satellite(df):
    sat_paths = {}
    for _, row in df.iterrows():
        p = fetch_satellite(row['lat'], row['lon'], row['id'])
        sat_paths[row['id']] = p
        time.sleep(0.1)
        print(f"Satellite done: {row['street']}")
    return sat_paths

Census Data

In [ ]:
def census_get(url, retries=3, timeout=30):
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=timeout)
            return r
        except (requests.ReadTimeout, requests.ConnectionError) as e:
            print(f"Attempt {attempt + 1}/{retries} failed: {e}")
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                print("All retries exhausted, skipping.")
                return None
    return None


def get_census_tract(lat, lon):
    url = (
        f"https://geocoding.geo.census.gov/geocoder/geographies/coordinates"
        f"?x={lon}&y={lat}&benchmark=Public_AR_Current&vintage=Census2020_Current"
        f"&layers=Census+Tracts&format=json"
    )
    r = census_get(url)
    if r is None or r.status_code != 200:
        print(f"Tract lookup failed")
        return None
    try:
        tract = r.json()['result']['geographies']['Census Tracts'][0]
        return {
            'state': tract['STATE'],
            'county': tract['COUNTY'],
            'tract': tract['TRACT']
        }
    except (KeyError, IndexError) as e:
        print(f"Tract parse error: {e}")
        return None


def get_census_demographics(state, county, tract):
    variables = ','.join([
        'B19013_001E', 'B25077_001E', 'B25064_001E',
        'B15003_022E', 'B15003_001E', 'B01003_001E',
        'B02001_002E', 'B02001_003E', 'B02001_005E', 'B03003_003E',
    ])
    url = (
        f"https://api.census.gov/data/2023/acs/acs5"
        f"?get={variables}"
        f"&for=tract:{tract}&in=state:{state}+county:{county}"
        f"&key={CENSUS_KEY}"
    )
    r = census_get(url)
    if r is None or r.status_code != 200:
        print(f"  Demographics failed")
        return None
    data = r.json()
    result = dict(zip(data[0], data[1]))

    total_pop  = int(result.get('B01003_001E') or 0)
    pop_25plus = int(result.get('B15003_001E') or 1)

    return {
        'census_median_income': result.get('B19013_001E'),
        'census_median_home_value': result.get('B25077_001E'),
        'census_median_rent': result.get('B25064_001E'),
        'census_pct_educated': round(int(result.get('B15003_022E') or 0) / pop_25plus, 4),
        'census_total_population': total_pop,
        'census_pct_white': round(int(result.get('B02001_002E') or 0) / max(total_pop, 1), 4),
        'census_pct_black': round(int(result.get('B02001_003E') or 0) / max(total_pop, 1), 4),
        'census_pct_asian': round(int(result.get('B02001_005E') or 0) / max(total_pop, 1), 4),
        'census_pct_hispanic': round(int(result.get('B03003_003E') or 0) / max(total_pop, 1), 4),
    }


def fetch_all_census(df):
    census_data = {}
    for _, row in df.iterrows():
        prop_id = str(row['id'])
        tract_info = get_census_tract(row['lat'], row['lon'])
        if tract_info:
            demo = get_census_demographics(
                tract_info['state'],
                tract_info['county'],
                tract_info['tract']
            )
            census_data[prop_id] = {**(tract_info or {}), **(demo or {})}
        else:
            census_data[prop_id] = {}
        time.sleep(1.0)
        print(f"  Census done: {row['street']}")
    return census_data

Saving eveyrthing

In [ ]:
def save(new_df):
    combined = pd.concat([existing_df, new_df], ignore_index=True)
    combined.drop_duplicates(subset='id', keep='first', inplace=True)
    combined.to_csv(MASTER_CSV, index=False)
    print(f"\nSaved {len(combined)} total properties to {MASTER_CSV}")

Main block to run everything

In [ ]:
if __name__ == '__main__':

    print("\n1. Fetching new listings...")
    new_props = fetch_all_new_listings(keyword='Boston, MA', listing_type='sold', max_pages=5)

    if not new_props:
        print("No new properties found. Exiting.")
    else:
        print(f"\nFound {len(new_props)} new properties")

        print("\n2. Parsing properties...")
        rows = parse_properties(new_props)
        new_df = pd.DataFrame(rows)
        new_df['id'] = new_df['id'].astype(str)
        save(new_df)
        print("Tabular data saved.")

        print("\n3. Fetching Street View images...")
        fetch_all_street_views(new_df)

        print("\n4. Fetching satellite images...")
        fetch_all_satellite(new_df)

        print("\n5. Fetching Census data...")
        census_data = fetch_all_census(new_df)
        census_df = pd.DataFrame.from_dict(census_data, orient='index')
        census_df.index.name = 'id'
        census_df = census_df.reset_index()
        census_df['id'] = census_df['id'].astype(str)
        new_df = new_df.merge(census_df, on='id', how='left')
        save(new_df)
        print(f"\nDone. Final shape: {new_df.shape}")

In [ ]:
def fill_missing_census(csv_path=MASTER_CSV):
    df = pd.read_csv(csv_path, dtype={'id': str})

    census_cols = ['census_median_income', 'census_median_home_value', 'census_median_rent',
                   'census_pct_educated',	'census_total_population',	'census_pct_white',	'census_pct_black',
                   'census_pct_asian', 'census_pct_hispanic']
    missing = df[df[census_cols].isnull().all(axis=1)].copy()
    print(f"Found {len(missing)} properties with missing census data")

    if missing.empty:
        print("Nothing to fill.")
        return df

    filled_count = 0
    for idx, row in missing.iterrows():
        print(f"\nRetrying: {row['street']}")
        time.sleep(2.0)

        tract_info = get_census_tract(row['lat'], row['lon'])
        if not tract_info:
            print(f"Tract lookup failed again, skipping")
            continue

        demo = get_census_demographics(
            tract_info['state'],
            tract_info['county'],
            tract_info['tract']
        )
        if not demo:
            print(f"Demographics failed again, skipping")
            continue

        # cast each value to match the column's existing dtype before writing
        for col, val in {**tract_info, **demo}.items():
            if col in df.columns:
                try:
                    val = df[col].dtype.type(val)
                except (ValueError, TypeError):
                    val = str(val)
            df.at[idx, col] = val

        filled_count += 1
        print(f"Filled successfully")
        df.to_csv(csv_path, index=False)

    print(f"\nDone. Filled {filled_count} of {len(missing)} properties")
    return df

df = fill_missing_census()

In [ ]:
import os
from pathlib import Path

GSV_DIR = Path('/content/drive/MyDrive/mmai_midterm_report/street_view')
SAT_DIR = Path('/content/drive/MyDrive/mmai_midterm_report/satellite')

df = pd.read_csv(MASTER_CSV, dtype={'id': str})

# satellite
df['sat_path'] = df['id'].apply(
    lambda pid: str(SAT_DIR / f"{pid}_sat.jpg")
    if (SAT_DIR / f"{pid}_sat.jpg").exists() else None
)

# street view
for heading in [0, 90, 180, 270]:
    df[f'gsv_path_{heading}'] = df['id'].apply(
        lambda pid: str(GSV_DIR / f"{pid}_h{heading}.jpg")
        if (GSV_DIR / f"{pid}_h{heading}.jpg").exists() else None
    )

df.to_csv(MASTER_CSV, index=False)

sat_found = df['sat_path'].notna().sum()
gsv_found = df['gsv_path_0'].notna().sum()
print(f"Satellite images matched: {sat_found} / {len(df)}")
print(f"Street view images matched: {gsv_found} / {len(df)}")

# show any properties missing images so we know what is missing
missing_sat = df[df['sat_path'].isna()][['id', 'street', 'city']]
missing_gsv = df[df['gsv_path_0'].isna()][['id', 'street', 'city']]

if not missing_sat.empty:
    print(f"\nMissing satellite ({len(missing_sat)}):")
    print(missing_sat.to_string(index=False))

if not missing_gsv.empty:
    print(f"\nMissing street view ({len(missing_gsv)}):")
    print(missing_gsv.to_string(index=False))

In [ ]:
import ast
import requests
import time
from pathlib import Path

PHOTOS_DIR = Path('/content/drive/MyDrive/mmai_midterm_report/interior_photos')
PHOTOS_DIR.mkdir(exist_ok=True)

df = pd.read_csv(MASTER_CSV, dtype={'id': str})

def download_photos(row):
    prop_id = row['id']

    # parse the photo_urls string back into a list
    try:
        urls = ast.literal_eval(row['photo_urls']) if pd.notna(row['photo_urls']) else []
    except (ValueError, SyntaxError):
        print(f"Could not parse photo_urls for {row['street']}")
        return 0

    prop_dir = PHOTOS_DIR / prop_id
    prop_dir.mkdir(exist_ok=True)

    downloaded = 0
    for i, url in enumerate(urls):
        path = prop_dir / f"{prop_id}_interior_{i}.jpg"
        if path.exists():
            continue
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200 and r.headers.get('content-type', '').startswith('image'):
                with open(path, 'wb') as f:
                    f.write(r.content)
                downloaded += 1
            time.sleep(0.05)
        except Exception as e:
            print(f"Failed {url}: {e}")

    return downloaded

total = 0
for _, row in df.iterrows():
    count = download_photos(row)
    total += count
    if count > 0:
        print(f"{row['street']}: {count} photos downloaded")

print(f"\nDone. {total} photos total across {len(df)} properties")
print(f"Saved to {PHOTOS_DIR}")